# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We'll walk through loading the dataset's schema, investigating its record sets and fields, extracting data by referencing entities via their `@id`, and performing basic exploratory data analysis.

### Dataset Source
The dataset source is provided as a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

See the [FAIR² metadata page](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for more info.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We'll display the `@id`, `name`, and other details for available record sets, their fields, and columns, to help us decide which ones to extract and analyze.

In [ ]:
# List all record sets, fields, and columns by their @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No accessible record sets were found in the metadata.")
else:
    print(f"Number of record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"=== Record Set: {rs['@id']} ===")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs["field"]:
                field_meta = dataset.metadata.find_entity(field["@id"]) if isinstance(field, dict) else dataset.metadata.find_entity(field)
                print(f"    • {field_meta['@id']} (name: {field_meta.get('name','')}, type: {field_meta.get('dataType','')})")
        elif 'fields' in rs:
            print("  Fields:")
            for field in rs['fields']:
                print(f"    • {field.get('@id','')} (name: {field.get('name','')})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

We'll demonstrate extracting records for all available record sets.

In [ ]:
# Prepare a list of record set @ids for extraction
all_record_set_ids = [rs["@id"] for rs in dataset.record_sets]

dataframes = {}
print("Extracting dataframes for each record set...\n")
for rs_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for record set {rs_id}")
        else:
            print(f"No records for {rs_id}")
    except Exception as ex:
        print(f"Error loading {rs_id}: {ex}")

if dataframes:
    # Show a preview of the first dataframe and its columns
    sample_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in {sample_rs_id}:")
    print(dataframes[sample_rs_id].columns.tolist())
    dataframes[sample_rs_id].head()
else:
    print('No record sets could be loaded as DataFrames.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field in one of the extracted DataFrames, filter records, perform normalization, and optionally group the data by a relevant attribute.

In [ ]:
# --- Pick a record set and a numeric field dynamically for demonstration ---
import numpy as np

if not dataframes:
    print("No data loaded for EDA.")
else:
    # Select the first loaded DataFrame
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    # Attempt to pick the first column with numeric dtype
    numeric_candidate = None
    for c in df.columns:
        # Try to convert column to float; ignore conversion errors
        # We'll check at least 5 non-null entries
        try:
            sample = pd.to_numeric(df[c], errors='coerce').dropna()
            if len(sample) > 5 and np.issubdtype(sample.dtype, np.number):
                numeric_candidate = c
                break
        except Exception: continue

    if numeric_candidate is None:
        print("No suitable numeric field found in the sample DataFrame.")
    else:
        print(f"Analyzing the numeric field: '{numeric_candidate}' in record set {rs_id}\n")
        # Convert column to float (for robust filtering/normalization)
        df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')
        threshold = df[numeric_candidate].mean() if df[numeric_candidate].mean() > 0 else 0
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > {threshold:.2f}:")
        display(filtered_df[[numeric_candidate]].head())

        # Normalize the field (z-score)
        filtered_df[f"{numeric_candidate}_normalized"] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
        print(f"\nNormalized {numeric_candidate} (first few rows):")
        display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Try grouping by another column (categorical)
        cat_field = None
        for col in df.columns:
            if col != numeric_candidate and df[col].dtype == 'object':
                num_unique = df[col].nunique(dropna=True)
                if 1 < num_unique < 10:
                    cat_field = col
                    break

        if cat_field is not None:
            grouped_df = filtered_df.groupby(cat_field)[numeric_candidate].mean()
            print(f"\nAverage {numeric_candidate} grouped by '{cat_field}':")
            display(grouped_df.head())
        else:
            print('\nNo suitable categorical field found for grouping.')

## 5. Visualization
Visualize numeric field distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or numeric_candidate is None:
    print('No suitable numeric field to plot.')
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_candidate].dropna(), kde=True, bins=30, color='cornflowerblue')
    plt.xlabel(numeric_candidate)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_candidate}')
    plt.tight_layout()
    plt.show()

    # Boxplot if a grouping field exists
    if cat_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=cat_field, y=numeric_candidate, data=df)
        plt.title(f'{numeric_candidate} by {cat_field}')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

* Load and parse a Croissant schema-driven dataset using the `mlcroissant` library,
* Explore record sets, fields, and columns with their `@id` references,
* Extract tabular data by referencing record set `@id`s and performing initial data wrangling,
* Conduct basic analysis such as filtering, normalization, and aggregation,
* Visualize results with standard plotting tools.

This provides a template for working with any Croissant-compatible dataset. For more detailed domain-specific analysis, review the dataset's schema or documentation for publishing conventions, field definitions, and best practices.